In [0]:
# Chemins des tables Silver (source) et Gold (destination)
catalog = "bixi_mobility"

silver_table = f"{catalog}.silver.trips_clean"

In [0]:
from pyspark.sql.functions import col, to_date, count, avg, round as spark_round

df_silver = spark.read.table(silver_table)

# Agrégation du nombre de trajets et de la durée moyenne, par jour
df_daily_trips = (
    df_silver
    .withColumn("trip_date", to_date("start_time"))
    .groupBy("trip_date")
    .agg(
        count("*").alias("nb_trips"),
        spark_round(avg("duration_sec"), 1).alias("avg_duration_sec")
    )
    .orderBy("trip_date")
)

# NB : cette table Gold est recalculée entièrement à chaque exécution
df_daily_trips.write.mode("overwrite").saveAsTable(f"{catalog}.gold.daily_trips")

In [0]:
# Popularité des stations en tant que point de départ
df_top_start_stations = (
    df_silver
    .groupBy("start_station_name", "start_borough")
    .agg(count("*").alias("nb_departures"))
    .orderBy(col("nb_departures").desc())
)

# Popularité des stations en tant que point d'arrivée
df_top_end_stations = (
    df_silver
    .groupBy("end_station_name", "end_borough")
    .agg(count("*").alias("nb_arrivals"))
    .orderBy(col("nb_arrivals").desc())
)

# Jointure des deux pour avoir une vue complète par station
df_station_popularity = (
    df_top_start_stations.alias("s")
    .join(
        df_top_end_stations.alias("e"),
        col("s.start_station_name") == col("e.end_station_name"),
        "outer"
    )
    .selectExpr(
        "coalesce(s.start_station_name, e.end_station_name) as station_name",
        "coalesce(s.start_borough, e.end_borough) as borough",
        "coalesce(nb_departures, 0) as nb_departures",
        "coalesce(nb_arrivals, 0) as nb_arrivals"
    )
    .withColumn("total_trips", col("nb_departures") + col("nb_arrivals"))
    .orderBy(col("total_trips").desc())
)

df_station_popularity.write.mode("overwrite").saveAsTable(f"{catalog}.gold.station_popularity")

In [0]:
from pyspark.sql.functions import dayofweek, hour, date_format

# Analyse des usages par jour de la semaine et par heure de la journée
# Utile pour repérer les pics de trafic (heures de pointe, week-ends vs semaine)
df_seasonality = (
    df_silver
    .withColumn("day_of_week", date_format("start_time", "EEEE"))
    .withColumn("hour_of_day", hour("start_time"))
    .groupBy("day_of_week", "hour_of_day")
    .agg(count("*").alias("nb_trips"))
    .orderBy("day_of_week", "hour_of_day")
)

df_seasonality.write.mode("overwrite").saveAsTable(f"{catalog}.gold.seasonality")

In [0]:
%sql
SELECT * FROM bixi_mobility.gold.daily_trips ORDER BY trip_date LIMIT 10;
SELECT * FROM bixi_mobility.gold.station_popularity ORDER BY total_trips DESC LIMIT 10;
SELECT * FROM bixi_mobility.gold.seasonality ORDER BY nb_trips DESC LIMIT 10;